# Modelo de celdas (B2)
Aprende a reconocer que es cada celda de un horario en Excel: `dia`, `hora`, `nombre`, `prueba` (clase de prueba, marca CP) u `otro`.
Datos: los Excels inventados de `datos/generar_excels.py` (ejecutalo antes). Rasgos: `rasgos_celdas.py` (el mismo codigo que usa el lector).
Separamos por archivo: 250 Excels para aprender y 50 de examen. Objetivo: >= 98% de celdas bien en el examen.

In [1]:
import sys
from pathlib import Path
import joblib
import pandas as pd
from openpyxl import load_workbook
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ))
from rasgos_celdas import ETIQUETAS, rasgos_hoja

GEN = RAIZ / "datos" / "generados"
etiquetas = pd.read_csv(GEN / "etiquetas.csv")
etiquetas.etiqueta.value_counts()

etiqueta
nombre    245498
hora       41968
otro       34097
prueba     18378
dia        10195
Name: count, dtype: int64

In [2]:
filas = []
for archivo in sorted(etiquetas.archivo.unique()):
    wb = load_workbook(GEN / "excels" / archivo, data_only=True)
    for ws in wb.worksheets:
        for f, c, _, r in rasgos_hoja(ws):
            filas.append({"archivo": archivo, "hoja": ws.title, "fila": f, "col": c, **r})
X = pd.DataFrame(filas).merge(etiquetas[["archivo", "hoja", "fila", "col", "etiqueta"]],
                              on=["archivo", "hoja", "fila", "col"], how="left", validate="one_to_one")
assert X.etiqueta.notna().all(), "Hay celdas sin etiqueta"
COLS = [c for c in X.columns if c not in ("archivo", "hoja", "fila", "col", "etiqueta")]
len(X), len(COLS)

(350136, 21)

In [3]:
archivos = sorted(X.archivo.unique())
entreno, examen = archivos[:250], archivos[250:]
A, B = X[X.archivo.isin(entreno)], X[X.archivo.isin(examen)]
modelo = RandomForestClassifier(n_estimators=60, max_depth=18, min_samples_leaf=2, n_jobs=-1, random_state=0)
modelo.fit(A[COLS], A.etiqueta)
pred = modelo.predict(B[COLS])
acierto = accuracy_score(B.etiqueta, pred)
print(f"Celdas bien clasificadas en el examen: {acierto:.2%} ({len(B)} celdas de {len(examen)} Excels)")
print(classification_report(B.etiqueta, pred, digits=4))
pd.DataFrame(confusion_matrix(B.etiqueta, pred, labels=ETIQUETAS), index=ETIQUETAS, columns=ETIQUETAS)

Celdas bien clasificadas en el examen: 100.00% (60451 celdas de 50 Excels)


              precision    recall  f1-score   support

         dia     1.0000    1.0000    1.0000      1695
        hora     1.0000    1.0000    1.0000      7303
      nombre     1.0000    1.0000    1.0000     42078
        otro     1.0000    0.9998    0.9999      6279
      prueba     1.0000    1.0000    1.0000      3096

    accuracy                         1.0000     60451
   macro avg     1.0000    1.0000    1.0000     60451
weighted avg     1.0000    1.0000    1.0000     60451



,dia,hora,nombre,prueba,otro
dia,1695,0,0,0,0
hora,0,7303,0,0,0
nombre,0,0,42078,0,0
prueba,0,0,0,3096,0
otro,0,0,1,0,6278


Algunos fallos, para ver donde se equivoca:

In [4]:
fallos = B.assign(pred=pred)[B.etiqueta != pred]
valores = etiquetas.set_index(["archivo", "hoja", "fila", "col"]).valor
fallos.assign(valor=[valores.get(k) for k in zip(fallos.archivo, fallos.hoja, fallos.fila, fallos.col)])[
    ["archivo", "valor", "etiqueta", "pred"]].head(15)

,archivo,valor,etiqueta,pred
340439,horario_291.xlsx,ver WhatsApp,otro,nombre


In [5]:
assert acierto >= 0.98, "No llega al 98%: no se guarda el modelo"
(RAIZ / "modelos").mkdir(exist_ok=True)
joblib.dump({"modelo": modelo, "columnas": COLS, "acierto_examen": acierto}, RAIZ / "modelos" / "modelo_celdas.joblib", compress=3)
print(f"Guardado: {(RAIZ / 'modelos' / 'modelo_celdas.joblib').stat().st_size / 1e6:.1f} MB")

Guardado: 0.4 MB
